In [66]:
import json
import re
import ast
import pandas as pd
from collections import Counter
import spacy

# ── 파일 경로 ──────────────────────────────────────────
DATASET_PATH   = 'hatexplain_prediction.csv'                        # HateXplain 원본
HURTLEX_PATH   = 'lexicons/hurtlex_EN.tsv'             # HurtLex 영어판
NRC_PATH       = 'NRC-Emotion-Lexicon/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt'
OUTPUT_PATH    = 'results/cell_a_anchors_v2_framed.csv'

LABEL_MAP = {0: 'hatespeech', 1: 'offensive', 2: 'normal'}

# spaCy (없으면: python -m spacy download en_core_web_sm)
nlp = spacy.load('en_core_web_sm')
print('✅ Libraries loaded.')

✅ Libraries loaded.


In [67]:
# ── HurtLex → SLUR_SET ────────────────────────────────
hurtlex = pd.read_csv(HURTLEX_PATH, sep='\t')
print('HurtLex categories:', hurtlex['category'].unique())  # 실제 카테고리 확인

TARGET_CATS = {'ps', 'rci', 'ddp', 'or', 'gi', 'qas', 're'} # AN/CV/AC 제외
slur_base = set(hurtlex[hurtlex['category'].isin(TARGET_CATS)]['lemma'].str.lower())

SLUR_SET = set(slur_base)
for s in list(slur_base):
    SLUR_SET.update({
        s+'s', s+'es', s+'z',
        s+'er', s+'ers', s+'ing', s+'ed'
    })
print(f'SLUR_SET size: {len(SLUR_SET)}')

# ── NRC Emotion Lexicon → POLARITY_CUE_LEXICON ─────────
nrc = pd.read_csv(NRC_PATH, sep='\t', header=None,
                  names=['word', 'emotion', 'association'])
neg_emotions = {'anger', 'disgust', 'fear'}
nrc_neg = nrc[nrc['emotion'].isin(neg_emotions) & (nrc['association'] == 1)]
word_counts = nrc_neg.groupby('word')['emotion'].count()
POLARITY_CUE_LEXICON = set(word_counts[word_counts >= 2].index)
print(f'POLARITY_CUE_LEXICON size: {len(POLARITY_CUE_LEXICON)}')

HurtLex categories: <StringArray>
['qas', 'cds',  'is',  're',  'om',  'an',  'or', 'svp', 'asm',  'pr', 'dmc',
 'asf', 'ddp',  'ps',  'pa', 'rci', 'ddf']
Length: 17, dtype: str
SLUR_SET size: 14804
POLARITY_CUE_LEXICON size: 1047


In [68]:
VALID_GROUP_TARGETS = {
    'African', 'Islam', 'Jewish', 'Women',
    'Refugee', 'Homosexual', 'Arab', 'Latino_Americans', 'Asian'
}

# ── 타겟 참조 토큰 (Filter 3 보조, target_tokens 생성용) ──
TARGET_REF = {
    # 중립 단어 + 주요 슬러/속어 포함 (혁오 발언에서 실제 사용되는 표현)
    'African':         [
        'black', 'african', 'negro', 'blacks',
        'nigger', 'niggers', 'nigga', 'niggas',
        'coon', 'coons', 'spook', 'spooks',
        'darkie', 'darkies', 'sambo',
    ],
    'Islam':           [
        'muslim', 'islam', 'islamic', 'muslims',
        'muzzie', 'muzzies', 'muzlim', 'muzrat',
        'moslem', 'moslems', 'muzzie', 'sandnigger',
        'jihadist', 'islamist',
    ],
    'Jewish':          [
        'jew', 'jewish', 'jews',
        'kike', 'kikes', 'yid', 'yids',
        'hebe', 'hebes', 'hymie',
    ],
    'Women':           [
        'woman', 'women', 'female', 'females', 'girl', 'girls',
        'bitch', 'bitches', 'slut', 'sluts',
        'whore', 'whores', 'hoe', 'hoes',
        'thot', 'cunt', 'cunts', 'dyke',
    ],
    'Refugee':         [
        'refugee', 'refugees', 'immigrant', 'immigrants', 'migrant', 'migrants',
        'illegal', 'illegals', 'invader', 'invaders',
        'asylum', 'foreigner', 'foreigners',
    ],
    'Homosexual':      [
        'gay', 'lesbian', 'homosexual', 'lgbt', 'queer',
        'faggot', 'faggots', 'fag', 'fags',
        'dyke', 'dykes', 'homo', 'homos',
        'tranny', 'sodomite', 'sodomites',
    ],
    'Arab':            [
        'arab', 'arabs', 'arabic',
        'sandnigger', 'towelhead', 'raghead',
        'camel jockey', 'terrorist',  # 맥락상 Arab 연결되는 경우
    ],
    'Latino_Americans':[
        'latino', 'latina', 'hispanic', 'latinos',
        'spic', 'spics', 'wetback', 'wetbacks',
        'beaner', 'beaners',
    ],
    'Asian':           [
        'asian', 'chinese', 'korean', 'japanese', 'asians',
        'chink', 'chinks', 'gook', 'gooks',
        'nip', 'nips', 'zipperhead', 'jap', 'japs',
        'curry', 'currynigger',
    ],
}

def get_target_tokens(tokens_lower, targets):
    found = []
    for t in targets:
        refs = TARGET_REF.get(t, [])
        found += [tok for tok in tokens_lower if tok in refs]
    return list(set(found))

def is_individual_attack(text, targets):
    doc = nlp(text[:200])
    persons = [ent.text for ent in doc.ents if ent.label_ == 'PERSON']
    return bool(persons) and targets == ['Women']

# ── 필터 순서대로 적용 ──────────────────────────────────
filtered = []

for _, row in df.iterrows():
    tokens     = row['tokens']
    ml         = row['majority_label']
    agreement  = row['agreement']
    targets    = row['targets']
    text       = ' '.join(tokens)
    toks_lower = [t.lower() for t in tokens]

    # Filter 1: hate or offensive, split 제외
    if ml not in ('hatespeech', 'offensive'):
        continue
    if agreement == 'split':
        continue

    # Filter 2: 토큰 수 10~60
    if not (10 <= len(tokens) <= 60):
        continue

    # Filter 3-1: Other 단독 제외
    specific = [t for t in targets if t != 'Other']
    if len(specific) == 0:
        continue

    # Filter 3-2: 유효 집단 타겟 존재
    valid_targets = [t for t in specific if t in VALID_GROUP_TARGETS]
    if not valid_targets:
        continue

    # Filter 3-3: 타겟 3개 초과 제외
    if len(specific) > 3:
        continue

    # Filter 4: 개인 공격 제외 (spaCy NER)
    if is_individual_attack(text, valid_targets):
        continue

    # Filter 5: 독립 cue ≥ 1
    slur_tokens     = [t for t in toks_lower if t in SLUR_SET]
    cue_tokens      = [t for t in toks_lower if t in POLARITY_CUE_LEXICON]
    non_slur_cue    = [t for t in cue_tokens if t not in SLUR_SET]
    slur_is_sole    = len(slur_tokens) > 0 and len(non_slur_cue) == 0
    if len(non_slur_cue) < 1:
        continue

    # Filter 6: NONE_DETECTED 프레이밍 제외
    framing = classify_framing(text, toks_lower)
    if framing == ['NONE_DETECTED']:
        continue

    # 메타데이터 수집 (Filter 7 전에 미리 계산)
    target_toks = get_target_tokens(toks_lower, valid_targets)

    # Filter 7: 명시적 타겟 언급 필터
    # 어노테이터가 타겟 표시했어도 텍스트에 없으면 Cell C 생성 시 타겟 소실 위험
    if len(target_toks) == 0:
        continue

    # Filter 8: 슬러 밀도 제어 (슬러 4개 이상 → Cell B 치환 부담 ↑, 소실 위험)
    if len(slur_tokens) > 3:
        continue

    filtered.append({
        'post_id':            row['post_id'],
        'text':               text,
        'text_clean':         text,
        'tokens':             tokens,
        'majority_label':     ml,
        'agreement':          agreement,
        'targets':            valid_targets,
        'cue_tokens':         cue_tokens,
        'slur_tokens':        slur_tokens,
        'non_slur_cue_tokens':non_slur_cue,
        'slur_is_sole_target':slur_is_sole,
        'target_tokens':      target_toks,
        'framing':            framing,
        'speech_act':         'STATEMENT',   # TODO: classify_speech_act() 구현 시 교체
        'primary_framing':    framing[0],
    })

df_filtered = pd.DataFrame(filtered)
print(f'After filters: {len(df_filtered):,} samples')
print(df_filtered['majority_label'].value_counts())
print(df_filtered['agreement'].value_counts())

After filters: 844 samples
majority_label
hatespeech    557
offensive     287
Name: count, dtype: int64
agreement
majority     476
unanimous    368
Name: count, dtype: int64


In [69]:
def parse_tokens(tok_str):
    """numpy repr ['u' 'really' ...] 형태 파싱 (콤마 없는 형식)"""
    tokens = re.findall(r"'([^']*)'|\"([^\"]*)\"", str(tok_str))
    return [a or b for a, b in tokens]

def parse_annotator_labels(ann_str):
    """'label': array([0, 2, 2]) 형태에서 라벨 리스트 추출"""
    match = re.search(r"'label':\s*array\(\[([^\]]+)\]\)", str(ann_str))
    if match:
        return [int(x.strip()) for x in match.group(1).split(',')]
    return []

def parse_targets(tgt_str):
    try:
        return ast.literal_eval(str(tgt_str))
    except:
        return []

df_raw = pd.read_csv(DATASET_PATH)
print(f'Loaded: {len(df_raw):,} rows')

records = []
for _, row in df_raw.iterrows():
    labels = parse_annotator_labels(row['annotators'])
    if not labels:
        continue

    ml, ml_count = Counter(labels).most_common(1)[0]
    unique_labels = set(labels)
    if ml_count == len(labels):
        agreement = 'unanimous'
    elif len(unique_labels) == len(labels):
        agreement = 'split'
    else:
        agreement = 'majority'

    tokens  = parse_tokens(row['post_tokens'])
    targets = parse_targets(row['targets'])

    records.append({
        'post_id':        row['post_id'],
        'tokens':         tokens,
        'majority_label': LABEL_MAP.get(ml, 'unknown'),
        'agreement':      agreement,
        'targets':        targets,
    })

df = pd.DataFrame(records)
print(f'Parsed: {len(df):,}')
print(df['majority_label'].value_counts())
print(df['agreement'].value_counts())

Loaded: 19,229 rows
Parsed: 19,229
majority_label
offensive     7814
hatespeech    5935
normal        5480
Name: count, dtype: int64
agreement
unanimous    9845
majority     9384
Name: count, dtype: int64


In [77]:
VALID_GROUP_TARGETS = {
    'African', 'Islam', 'Jewish', 'Women',
    'Refugee', 'Homosexual', 'Arab', 'Latino_Americans', 'Asian'
}

# ── 타겟 참조 토큰 (Filter 3 보조, target_tokens 생성용) ──
TARGET_REF = {
    # 중립 단어 + 주요 슬러/속어 포함 (혁오 발언에서 실제 사용되는 표현)
    'African':         [
        'black', 'african', 'negro', 'blacks',
        'nigger', 'niggers', 'nigga', 'niggas',
        'coon', 'coons', 'spook', 'spooks',
        'darkie', 'darkies', 'sambo',
    ],
    'Islam':           [
        'muslim', 'islam', 'islamic', 'muslims',
        'muzzie', 'muzzies', 'muzlim', 'muzrat',
        'moslem', 'moslems', 'muzzie', 'sandnigger',
        'jihadist', 'islamist',
    ],
    'Jewish':          [
        'jew', 'jewish', 'jews',
        'kike', 'kikes', 'yid', 'yids',
        'hebe', 'hebes', 'hymie',
    ],
    'Women':           [
        'woman', 'women', 'female', 'females', 'girl', 'girls',
        'bitch', 'bitches', 'slut', 'sluts',
        'whore', 'whores', 'hoe', 'hoes',
        'thot', 'cunt', 'cunts', 'dyke',
    ],
    'Refugee':         [
        'refugee', 'refugees', 'immigrant', 'immigrants', 'migrant', 'migrants',
        'illegal', 'illegals', 'invader', 'invaders',
        'asylum', 'foreigner', 'foreigners',
    ],
    'Homosexual':      [
        'gay', 'lesbian', 'homosexual', 'lgbt', 'queer',
        'faggot', 'faggots', 'fag', 'fags',
        'dyke', 'dykes', 'homo', 'homos',
        'tranny', 'sodomite', 'sodomites',
    ],
    'Arab':            [
        'arab', 'arabs', 'arabic',
        'sandnigger', 'towelhead', 'raghead',
        'camel jockey', 'terrorist',  # 맥락상 Arab 연결되는 경우
    ],
    'Latino_Americans':[
        'latino', 'latina', 'hispanic', 'latinos',
        'spic', 'spics', 'wetback', 'wetbacks',
        'beaner', 'beaners',
    ],
    'Asian':           [
        'asian', 'chinese', 'korean', 'japanese', 'asians',
        'chink', 'chinks', 'gook', 'gooks',
        'nip', 'nips', 'zipperhead', 'jap', 'japs',
        'curry', 'currynigger',
    ],
}

def get_target_tokens(tokens_lower, targets):
    found = []
    for t in targets:
        refs = TARGET_REF.get(t, [])
        found += [tok for tok in tokens_lower if tok in refs]
    return list(set(found))

def is_individual_attack(text, targets):
    doc = nlp(text[:200])
    persons = [ent.text for ent in doc.ents if ent.label_ == 'PERSON']
    return bool(persons) and targets == ['Women']

# ── 필터 순서대로 적용 ──────────────────────────────────
filtered = []

for _, row in df.iterrows():
    tokens     = row['tokens']
    ml         = row['majority_label']
    agreement  = row['agreement']
    targets    = row['targets']
    text       = ' '.join(tokens)
    toks_lower = [t.lower() for t in tokens]

    # Filter 1: hate or offensive, split 제외
    if ml not in ('hatespeech', 'offensive'):
        continue
    if agreement == 'split':
        continue

    # Filter 2: 토큰 수 10~60
    if not (10 <= len(tokens) <= 60):
        continue

    # Filter 3-1: Other 단독 제외
    specific = [t for t in targets if t != 'Other']
    if len(specific) == 0:
        continue

    # Filter 3-2: 유효 집단 타겟 존재
    valid_targets = [t for t in specific if t in VALID_GROUP_TARGETS]
    if not valid_targets:
        continue

    # Filter 3-3: 타겟 3개 초과 제외
    if len(specific) > 3:
        continue

    # Filter 4: 개인 공격 제외 (spaCy NER)
    if is_individual_attack(text, valid_targets):
        continue

    # Filter 5: 독립 cue ≥ 1
    slur_tokens     = [t for t in toks_lower if t in SLUR_SET]
    cue_tokens      = [t for t in toks_lower if t in POLARITY_CUE_LEXICON]
    non_slur_cue    = [t for t in cue_tokens if t not in SLUR_SET]
    slur_is_sole    = len(slur_tokens) > 0 and len(non_slur_cue) == 0
    if len(non_slur_cue) < 1:
        continue

    # Filter 6: NONE_DETECTED 프레이밍 제외
    framing = classify_framing(text, toks_lower)
    if framing == ['NONE_DETECTED']:
        continue

    # 메타데이터 수집 (Filter 7 전에 미리 계산)
    target_toks = get_target_tokens(toks_lower, valid_targets)

    # Filter 7: 명시적 타겟 언급 필터
    # 어노테이터가 타겟 표시했어도 텍스트에 없으면 Cell C 생성 시 타겟 소실 위험
    if len(target_toks) == 0:
        continue

    # Filter 8: 제거 — African/Homosexual 등 unanimous 샘플에서 슬러 여러 개 쓰는 경향 있음
    # 슬러 밀도 제어는 Cell B 프롬프트 레벨에서 처리

    filtered.append({
        'post_id':            row['post_id'],
        'text':               text,
        'text_clean':         text,
        'tokens':             tokens,
        'majority_label':     ml,
        'agreement':          agreement,
        'targets':            valid_targets,
        'cue_tokens':         cue_tokens,
        'slur_tokens':        slur_tokens,
        'non_slur_cue_tokens':non_slur_cue,
        'slur_is_sole_target':slur_is_sole,
        'target_tokens':      target_toks,
        'framing':            framing,
        'speech_act':         'STATEMENT',   # TODO: classify_speech_act() 구현 시 교체
        'primary_framing':    framing[0],
    })

df_filtered = pd.DataFrame(filtered)
print(f'After filters: {len(df_filtered):,} samples')
print(df_filtered['majority_label'].value_counts())
print(df_filtered['agreement'].value_counts())

After filters: 1,065 samples
majority_label
hatespeech    743
offensive     322
Name: count, dtype: int64
agreement
majority     573
unanimous    492
Name: count, dtype: int64


In [78]:
CAP = {
    'African':          200,
    'Women':            200,
    'Islam':            200,
    'Jewish':           180,
    'Refugee':          100,
    'Homosexual':       100,
    'Arab':              80,
    'Asian':             50,
    'Latino_Americans': 100,   # cap 미정 → 일단 100
}

sampled_ids = set()
result_rows = []

for target, cap in CAP.items():
    sub = df_filtered[df_filtered['targets'].apply(lambda ts: target in ts)].copy()
    # unanimous 우선
    unanimous = sub[sub['agreement'] == 'unanimous']
    majority  = sub[sub['agreement'] == 'majority']

    chosen = unanimous.sample(n=min(len(unanimous), cap), random_state=42)
    remaining = cap - len(chosen)
    if remaining > 0:
        majority_not_chosen = majority[~majority['post_id'].isin(chosen['post_id'])]
        extra = majority_not_chosen.sample(n=min(len(majority_not_chosen), remaining), random_state=42)
        chosen = pd.concat([chosen, extra])

    new = chosen[~chosen['post_id'].isin(sampled_ids)]
    sampled_ids.update(new['post_id'].tolist())
    result_rows.append(new)
    print(f'{target}: {len(new)} samples (cap {cap})')

df_final = pd.concat(result_rows).drop_duplicates(subset='post_id').reset_index(drop=True)
print(f'\n✅ Final Cell A: {len(df_final):,} samples')
print(df_final['majority_label'].value_counts())
print(df_final['agreement'].value_counts())

African: 200 samples (cap 200)
Women: 157 samples (cap 200)
Islam: 168 samples (cap 200)
Jewish: 121 samples (cap 180)
Refugee: 64 samples (cap 100)
Homosexual: 52 samples (cap 100)
Arab: 12 samples (cap 80)
Asian: 16 samples (cap 50)
Latino_Americans: 0 samples (cap 100)

✅ Final Cell A: 790 samples
majority_label
hatespeech    589
offensive     201
Name: count, dtype: int64
agreement
unanimous    492
majority     298
Name: count, dtype: int64


In [79]:
# 저장
df_final.to_csv(OUTPUT_PATH, index=False)
print(f'✅ Saved → {OUTPUT_PATH}')
df_final.head(3)

✅ Saved → results/cell_a_anchors_v2_framed.csv


,post_id,text,text_clean,tokens,majority_label,agreement,targets,cue_tokens,slur_tokens,non_slur_cue_tokens,slur_is_sole_target,target_tokens,framing,speech_act,primary_framing
0,5689010_gab,i honestly believe europe was jealous of our n...,i honestly believe europe was jealous of our n...,"[i, honestly, believe, europe, was, jealous, o...",hatespeech,unanimous,"[African, Islam]","[jealous, violent]","[niggers, retarded, violent, niggers]",[jealous],False,"[niggers, muslims]","[INTELLECTUAL_INFERIORITY, CRIMINAL_DANGER]",STATEMENT,INTELLECTUAL_INFERIORITY
1,8279790_gab,i only ask because if it a nigger than i care ...,i only ask because if it a nigger than i care ...,"[i, only, ask, because, if, it, a, nigger, tha...",hatespeech,unanimous,"[African, Jewish]",[bad],"[nigger, nonsense, jew]",[bad],False,"[nigger, jew]",[THREAT_VIOLENCE],STATEMENT,THREAT_VIOLENCE
2,24455415_gab,pennies kick flippin in a garbage can will not...,pennies kick flippin in a garbage can will not...,"[pennies, kick, flippin, in, a, garbage, can, ...",hatespeech,unanimous,[African],[slaughter],"[garbage, coons]",[slaughter],False,[coons],"[DEHUMANIZATION, THREAT_VIOLENCE]",STATEMENT,DEHUMANIZATION


In [80]:
print('=== 타겟별 분포 ===')
from collections import Counter
target_counter = Counter()
for ts in df_final['targets']:
    for t in ts:
        target_counter[t] += 1
for k, v in sorted(target_counter.items(), key=lambda x: -x[1]):
    print(f'  {k:20s}: {v}')

print('\n=== 프레이밍별 분포 (primary_framing) ===')
print(df_final['primary_framing'].value_counts())

print('\n=== agreement 분포 ===')
print(df_final['agreement'].value_counts())


=== 타겟별 분포 ===
  African             : 241
  Islam               : 226
  Women               : 207
  Jewish              : 190
  Refugee             : 131
  Homosexual          : 101
  Arab                : 90
  Asian               : 23

=== 프레이밍별 분포 (primary_framing) ===
primary_framing
THREAT_VIOLENCE             189
CRIMINAL_DANGER             105
DEHUMANIZATION              102
SEXUAL_GENDERED              96
CONSPIRACY                   73
INTELLECTUAL_INFERIORITY     71
MORAL_DISGUST                69
EXCLUSION                    54
RELIGIOUS                    13
GENERALIZATION               10
ECONOMIC_BURDEN               8
Name: count, dtype: int64

=== agreement 분포 ===
agreement
unanimous    492
majority     298
Name: count, dtype: int64


In [81]:
# ── 1. RELIGIOUS primary_framing 샘플 확인 (오분류 점검)
print('=== RELIGIOUS primary_framing 예시 ===')
rel = df_final[df_final['primary_framing'] == 'RELIGIOUS']
for _, r in rel.head(10).iterrows():
    print(f"[non_slur_cue] {r['non_slur_cue_tokens']}")
    print(f"  {r['text'][:100]}")
    print()

# ── 2. 프레이밍별 평균 토큰 수
df_final['token_len'] = df_final['tokens'].apply(len)
print('=== 프레이밍별 평균 토큰 수 ===')
print(df_final.groupby('primary_framing')['token_len'].mean().round(1).sort_values())

# ── 3. 타겟 × primary_framing 교차표
print('\n=== 타겟 × primary_framing 교차표 ===')
df_exploded = df_final.copy()
df_exploded = df_exploded.explode('targets').reset_index(drop=True)  # 중복 인덱스 제거
cross = pd.crosstab(df_exploded['targets'], df_exploded['primary_framing'])
print(cross)

# ── 4. non_slur_cue 토큰 수 분포
df_final['num_non_slur'] = df_final['non_slur_cue_tokens'].apply(len)
print('\n=== non_slur_cue 토큰 수 분포 ===')
print(df_final['num_non_slur'].value_counts().sort_index())

# ── 5. 복합 프레이밍(2개 이상) 비율
df_final['num_framing'] = df_final['framing'].apply(len)
print('\n=== 프레이밍 개수 분포 ===')
print(df_final['num_framing'].value_counts().sort_index())
print(f"복합 프레이밍 비율: {(df_final['num_framing'] >= 2).mean():.1%}")

=== RELIGIOUS primary_framing 예시 ===
[non_slur_cue] ['cannon']
  but violating it will not look at the entire structure of censorship sjws need the <number> billion 

[non_slur_cue] ['infidel', 'suicide']
  you mean nobodies rushing in to buy muslim ms marvel or titty bar thor the moslem green lantern oath

[non_slur_cue] ['misconception']
  oh muslim leaders of today do you think that you will be not asked about not fulfilling your duty of

[non_slur_cue] ['hate']
  ha ha ha reminds me of the hate mail from calgary being sent to synagogues of satan this kike commie

[non_slur_cue] ['communism', 'evil', 'opposed']
  i have done my heavy thinking and reached the following conclusion fascism communism are evil twenti

[non_slur_cue] ['armed']
  arm yourself against soros funded altifa tyranny and muslime ohell sedition there is no place in ame

[non_slur_cue] ['socialist', 'traitor', 'suicide']
  following on from the other socialist tory pm the equally useless europhile traitor camoron 

In [82]:
# GENERALIZATION primary_framing 샘플 — 실제로 일반화 표현인지 확인
print('=== GENERALIZATION primary_framing 예시 ===')
gen = df_final[df_final['primary_framing'] == 'GENERALIZATION']
for _, r in gen.head(10).iterrows():
    print(f"  {r['text'][:100]}")
    print()

# primary_framing == GENERALIZATION 인데 THREAT_VIOLENCE도 있는 케이스 확인
both = gen[gen['framing'].apply(lambda f: 'THREAT_VIOLENCE' in f)]
print(f'GENERALIZATION이지만 THREAT_VIOLENCE도 포함: {len(both)}건')
for _, r in both.head(5).iterrows():
    print(f"  framing={r['framing']} | {r['text'][:80]}")
    print()

=== GENERALIZATION primary_framing 예시 ===
  fuck all you gay bitches that are gonna buy the codename elf bundle kys niggers

  not for long boomer the kikes are gonna make sure they are all in spanish then they are gonna make y

  no most people are good people do you think all men are evil maybe because you have been victimized 

  top of it bitch happy sunday i hope you all make it to church i will not but i hope you all do

  hate to burst your bubble pal but no woman of any racial ethnicity are superior if any all women of 

  you all be wanting a real bitch then do not know what to do wit her ✌ 🏽

  where are muh 9 9 redballons did you all bitch up again i know you are reading this alt kikes

  this is why all jews are hungry to confiscate weapons because for generations their accursed race ha

  all jews world wide are about to have everyone in proximity attacking them

  i hate arabs why they said we all are working in macdonalds عمال_ماك

GENERALIZATION이지만 THREAT_VIOLENCE도 포함: 

In [76]:
# 첫 번째 RELIGIOUS 샘플 전체 텍스트 확인
rel = df_final[df_final['primary_framing'] == 'RELIGIOUS']
row = rel.iloc[0]
print('full text:', row['text'])
print('framing:', row['framing'])
print('slur_tokens:', row['slur_tokens'])

full text: you mean nobodies rushing in to buy muslim ms marvel or titty bar thor the moslem green lantern oath in brightest day in blackest night no unattended 6 yo infidel girl shall escape my sight let all who worship civilizations might behold my green jihadi suicide vest light
framing: ['RELIGIOUS']
slur_tokens: ['green', 'blackest', 'green']
